# Coaid Unified Schema Process

This is where all the steps are taken to convert the Liar dataset to the agreed unified schema.

Unified Schema:
- `dataset`, `id`, `split`
- `label_raw`, `label` (true/mixed/false), `label_confidence` ∈ {gold, weak}, `label_raw_source`
- **Short text**: `claim_text` (tweet, post, or title)
- **Long text (optional)**: `article_text` (parsed), `content_status` ∈ {full_article, partial, title_only}
- URLs/meta: `post_url`/`news_url`, `archive_url`, `is_archived`, `source_domain`

In [1]:
import numpy as np
import pandas as pd
from data.unified_schema.coaid.article_scraping import threaded_hydrate, hydrate_claim

## Load all datasets and concat them across dates

In [2]:
claim_fake_05 = pd.read_csv("../../raw/coaid/05-01-2020/ClaimFakeCOVID-19.csv")
claim_fake_07 = pd.read_csv("../../raw/coaid/07-01-2020/ClaimFakeCOVID-19.csv")

claim_fake = pd.concat([claim_fake_05,claim_fake_07],axis=0,ignore_index=True)

In [3]:
claim_real_05 = pd.read_csv("../../raw/coaid/05-01-2020/ClaimRealCOVID-19.csv")
claim_real_07 = pd.read_csv("../../raw/coaid/07-01-2020/ClaimRealCOVID-19.csv")
claim_real_09 = pd.read_csv("../../raw/coaid/09-01-2020/ClaimRealCOVID-19.csv")
claim_real_11 = pd.read_csv("../../raw/coaid/11-01-2020/ClaimRealCOVID-19.csv")

claim_real = pd.concat([claim_real_05,claim_real_07,claim_real_09,claim_real_11],axis=0,ignore_index=True)

In [4]:
news_fake_05 = pd.read_csv("../../raw/coaid/05-01-2020/NewsFakeCOVID-19.csv")
news_fake_07 = pd.read_csv("../../raw/coaid/07-01-2020/NewsFakeCOVID-19.csv")
news_fake_09 = pd.read_csv("../../raw/coaid/09-01-2020/NewsFakeCOVID-19.csv")
news_fake_11 = pd.read_csv("../../raw/coaid/11-01-2020/NewsFakeCOVID-19.csv")

news_fake = pd.concat([news_fake_05,news_fake_07,news_fake_09,news_fake_11],axis=0,ignore_index=True)

In [5]:
news_real_05 = pd.read_csv("../../raw/coaid/05-01-2020/NewsRealCOVID-19.csv")
news_real_07 = pd.read_csv("../../raw/coaid/07-01-2020/NewsRealCOVID-19.csv")
news_real_09 = pd.read_csv("../../raw/coaid/09-01-2020/NewsRealCOVID-19.csv")
news_real_11 = pd.read_csv("../../raw/coaid/11-01-2020/NewsRealCOVID-19.csv")

news_real = pd.concat([news_real_05,news_real_07,news_real_09,news_real_11],axis=0,ignore_index=True)

## Unify Schema

### Claims

In [6]:
claim_real['label'] = 'true'
claim_fake['label'] = 'false'

In [7]:
claim_fake.columns

Index(['Unnamed: 0', 'fact_check_url', 'news_url', 'title', 'label'], dtype='object')

In [8]:
claim_real.columns

Index(['Unnamed: 0', 'fact_check_url', 'news_url', 'title', 'label'], dtype='object')

In [9]:
claim = pd.concat([claim_fake,claim_real],axis=0,ignore_index=True)
claim.head()

,Unnamed: 0,fact_check_url,news_url,title,label
0,100000,medicalnewstoday.com,https://www.medicalnewstoday.com/articles/coro...,"""Spraying chlorine or alcohol on the skin kill...",false
1,100001,medicalnewstoday.com,https://www.medicalnewstoday.com/articles/coro...,"""Only older adults and young people are at risk""",false
2,100002,medicalnewstoday.com,https://www.medicalnewstoday.com/articles/coro...,"""Children cannot get COVID-19""",false
3,100003,medicalnewstoday.com,https://www.medicalnewstoday.com/articles/coro...,"""COVID-19 is just like the flu""",false
4,100004,medicalnewstoday.com,https://www.medicalnewstoday.com/articles/coro...,"""Everyone with COVID-19 dies""",false


In [13]:
def claim_schema(
    df: pd.DataFrame,
    split: str = "train",
    dataset: str = "coaid",
    title_col: str = "title",
    url_col: str = "news_url",
    max_workers: int = 12,
    batch_size = None,
    save_batches_dir = None,
    throttle_seconds: float = 1.0,
    show_progress: bool = True
) -> pd.DataFrame:
    """
    Minimal wrapper to:
      - preserve all original columns in `df`,
      - add only the URL/article-related columns produced by threaded_hydrate,
      - avoid overwriting any existing original columns.

    Parameters:
      - df: input DataFrame (will not be modified in-place)
      - title_col / url_col: column names for title and url in the input frame
      - other params forwarded to threaded_hydrate
    """
    # shallow copy to avoid mutating caller's frame
    temp = df.copy()

    # canonical metadata columns
    temp["dataset"] = dataset
    temp["split"] = split

    # drop legacy unnamed index column if present (safe)
    temp.drop(columns=["Unnamed: 0"], inplace=True, errors="ignore")

    # set label confidence
    temp["label_confidence"] = "gold"

    # Call threaded_hydrate to get URL/article metadata.
    # threaded_hydrate returns a DataFrame indexed by the original df index.
    res_df = threaded_hydrate(
        temp,
        title_col=title_col,
        url_col=url_col,
        hydrate_fn=None,
        dataset=dataset,
        max_workers=max_workers,
        batch_size=batch_size,
        save_batches_dir=save_batches_dir,
        throttle_seconds=throttle_seconds,
        show_progress=show_progress,
    )

    # Fields we expect from hydrate_claim (defensive)
    expected_cols = [
        "article_text",
        "content_status",
        "news_url",
        "archive_url",
        "is_archived",
        "source_domain",
        "is_hydrated",
        "fetch_status",
        "lang",
        "content_char_len",
        "claim_norm_hash",
        "ingested_at",
        "fetch_attempts",
        "last_fetch_at",
    ]

    # Keep only the expected columns that exist in res_df
    available_cols = [c for c in expected_cols if c in res_df.columns]

    # IMPORTANT: do not overwrite any original columns in temp.
    # Only add columns that are not already present in the original frame.
    cols_to_add = [c for c in available_cols if c not in temp.columns]

    # Align res_df to temp's index, then join only the desired columns
    res_df = res_df.reindex(temp.index)
    out = temp.join(res_df[cols_to_add], how="left")

    # Fill sensible defaults for the newly added columns (if present)
    if "content_status" in out.columns:
        out["content_status"] = out["content_status"].fillna("title_only")
    if "fetch_status" in out.columns:
        out["fetch_status"] = out["fetch_status"].fillna("not_fetched")
    if "content_char_len" in out.columns:
        out["content_char_len"] = out["content_char_len"].fillna(0).astype(int)
    if "is_hydrated" in out.columns:
        out["is_hydrated"] = out["is_hydrated"].fillna(False).astype(bool)
    if "is_archived" in out.columns:
        out["is_archived"] = out["is_archived"].fillna(False).astype(bool)
    if "source_domain" in out.columns:
        out["source_domain"] = out["source_domain"].fillna("")

    out.rename(columns={"title": "claim_text"}, inplace=True)
    # Return augmented DataFrame; original input columns remain unchanged.
    return out

In [14]:
claim_final = claim_schema(claim,'train','coaid')

  0%|          | 0/518 [00:00<?, ?it/s]

In [16]:
claim_final.head()

,fact_check_url,news_url,claim_text,label,dataset,split,label_confidence,article_text,content_status,archive_url,is_archived,source_domain,is_hydrated,fetch_status,lang,content_char_len,claim_norm_hash,ingested_at,fetch_attempts,last_fetch_at
0,medicalnewstoday.com,https://www.medicalnewstoday.com/articles/coro...,"""Spraying chlorine or alcohol on the skin kill...",false,coaid,train,gold,Share on Pinterest Design by Andrew Nguyen\n\n...,partial,None,False,www.medicalnewstoday.com,True,success,en,14363,3c009e65f139be1e301f16445d3f4b629c318438,2025-11-11T00:46:54.803951+00:00,1,2025-11-11T00:46:55.314652+00:00
1,medicalnewstoday.com,https://www.medicalnewstoday.com/articles/coro...,"""Only older adults and young people are at risk""",false,coaid,train,gold,Share on Pinterest Design by Andrew Nguyen\n\n...,partial,None,False,www.medicalnewstoday.com,True,success,en,14363,cc07271af27f0283b641237511bdebd97233a76f,2025-11-11T00:46:55.809775+00:00,1,2025-11-11T00:46:56.981267+00:00
2,medicalnewstoday.com,https://www.medicalnewstoday.com/articles/coro...,"""Children cannot get COVID-19""",false,coaid,train,gold,Share on Pinterest Design by Andrew Nguyen\n\n...,partial,None,False,www.medicalnewstoday.com,True,success,en,14363,b68387a76bf777c9b398e22fdc437cf097764b9e,2025-11-11T00:46:55.805392+00:00,1,2025-11-11T00:46:56.980906+00:00
3,medicalnewstoday.com,https://www.medicalnewstoday.com/articles/coro...,"""COVID-19 is just like the flu""",false,coaid,train,gold,Share on Pinterest Design by Andrew Nguyen\n\n...,partial,None,False,www.medicalnewstoday.com,True,success,en,14363,83c391e0dbefe0b59203d12b038ad48ec2c7e772,2025-11-11T00:46:55.809347+00:00,1,2025-11-11T00:46:56.863263+00:00
4,medicalnewstoday.com,https://www.medicalnewstoday.com/articles/coro...,"""Everyone with COVID-19 dies""",false,coaid,train,gold,Share on Pinterest Design by Andrew Nguyen\n\n...,partial,None,False,www.medicalnewstoday.com,True,success,en,14363,dcb79cec8ac3c34201f1c4e7d892606586eff268,2025-11-11T00:46:55.810823+00:00,1,2025-11-11T00:46:56.980176+00:00


This is an example article_text output

Share on Pinterest Design by Andrew Nguyen\n\nThis article was updated on November 8, 2020\n\nThe novel coronavirus, SARS-CoV-2, has spread from Wuhan, China, to every continent except Antarctica.\n\nThe World Health Organization (WHO) changed their classification of the situation from a public health emergency of international concern to a pandemic on March 11, 2020.\n\nThe virus has been responsible for tens of millions of infections globally, causing more than a million deaths. The United States has been the most affected country.\n\nAs ever, when the word “pandemic” began appearing in headlines, people became fearful — and with fear came misinformation and rumors.\n\nBelow, we dissect some of the most common myths currently circulating on social media and beyond.\n\nMedical Myths Coronavirus resources For more advice on COVID-19 prevention and treatment, visit our coronavirus hub.\n\n1. Spraying chlorine or alcohol on the skin kills viruses in the body\n\nApplying alcohol or chlorine to the skin can cause harm, especially if it enters the eyes or mouth. These chemicals can disinfect surfaces, but people should not use them on their bodies.\n\nAlso, these products cannot kill viruses inside the body.\n\n2. Only older adults and people with preexisting conditions are at risk of infections and complications\n\nSARS-CoV-2, like other coronaviruses, can transmit to people of any age. However, older adults and individuals with preexisting health conditions, such as diabetes, obesity, or asthma, are more likely to become severely ill.\n\nWhile people under 40 , including children, are less likely to become severely ill with COVID-19, the disease can lead to complications and death in anyone.\n\n3. Children cannot get COVID-19\n\nAnyone, of any age, can develop the infection that causes COVID-19.\n\nSo far, most COVID-19 cases have been in adults, but children are not immune. That said, most children who develop COVID-19 have mild symptoms or none at all.\n\nAlso, on May 15, 2020, the WHO released a commentary about an inflammatory condition in children and adolescents that may have links with COVID-19.\n\nScientists currently know little about this condition, but research from May suggests that it is rare, “probably affecting no more than 1 in 1,000 children exposed to SARS-CoV-2.”\n\n4. COVID-19 is just like the flu\n\nInfection with the virus SARS-CoV-2 can cause COVID-19, an illness that can cause flu-like symptoms, such as body aches, a fever, and a cough. Symptoms of either COVID-19 or the flu can be mild, severe, or rarely, fatal. Both illnesses can also cause pneumonia.\n\nHowever, the overall profile of COVID-19 is more serious. Different countries have reported different mortality rates, and the case fatality rate in the U.S. appears to be around 2.6%.\n\nWhile scientists are still determining the exact mortality rate based on developing data, it is likely to be many times higher than that of the seasonal flu.\n\n5. Everyone with COVID-19 dies\n\nThis is false. As we explain above, COVID-19 is fatal for a small percentage of people who develop the illness.\n\nThe WHO have reported that around 80% of people with COVID-19 experience a relatively form of the illness and do not need specialist treatment in a hospital. Mild symptoms may include a fever, a cough, a sore throat, tiredness, and shortness of breath.\n\nAlso, many people with the underlying infection experience no symptoms.\n\n6. Cats and dogs spread the coronavirus\n\nThere have been several reports of pets developing the infection, including in the U.S. In most cases, the pets became sick after coming into contact with people who had COVID-19.\n\nAccording to the Centers for Disease Control and Prevention (CDC) , “There is no evidence that animals play a significant role in spreading the virus that causes COVID-19.”\n\nScientists are debating the importance of these cases in animals. For instance, Jonathan Ball, a professor of molecular virology at the University of Nottingham, in the United Kingdom, says:\n\n“We have to differentiate between real infection and just detecting the presence of the virus. I still think it’s questionable how relevant it is to the human outbreak, as most of the global outbreak has been driven by human-to-human transmission.”\n\n7. Face masks always protect against the coronavirus\n\nHealthcare workers use professional face masks that fit tightly to protect themselves from infections.\n\nDisposable and cloth masks can protect against droplets, but neither can protect against aerosolized particles.\n\nThe CDC recommend that all people wear cloth face masks in public areas where it is difficult to maintain a 6-foot, or 2-meter, distance from others. This helps slow the spread of the virus.\n\nEven while wearing a mask, it is essential to continue with other precautions, such as not touching the face, physical distancing, and washing the hands frequently.\n\nInstructions for making masks at home are available here .\n\nSurgical masks and N95 respirators provide greater protection, but reserve these for healthcare workers.\n\n8. Hand dryers kill the coronavirus\n\nHand dryers do not kill SARS-CoV-2. The best way to protect oneself and others from the virus is to wash the hands with soap and water frequently for at least 20 seconds at a time.\n\nWhen this is not possible, use an alcohol-based hand sanitizer.\n\n9. SARS-CoV-2 is just a mutated form of the common cold virus\n\nCoronaviruses are a large family, and each has spiky proteins on their surface. Some use humans as their primary host and cause the common cold.\n\n\n\nOther coronaviruses, including SARS-CoV-2, primarily infect animals.\n\nLike COVID-19, Middle East respiratory syndrome (MERS) and severe acute respiratory syndrome (SARS) are caused by coronaviruses. These viruses also initially passed to humans from animals.\n\n10. You have to be with someone for 10 minutes to catch the virus\n\nThe longer a person is close to someone with the infection, the likelier the virus is to transmit. However, the virus can pass from person to person in under 10 minutes.\n\n11. Rinsing the nose with saline protects against the coronavirus\n\nThere is no evidence that a saline nasal rinse protects against any respiratory infections.\n\n\n\nSome research suggests that a rinse might ease the symptoms of acute upper respiratory tract infections, but scientists have not found that this technique reduces the risk of infection.\n\n12. You can protect yourself by gargling bleach\n\nPeople should never put bleach in their mouths. Gargling bleach could never benefit a person’s health.\n\n\n\nBleach is corrosive and can cause serious damage.\n\n13. Antibiotics kill the coronavirus\n\nAntibiotics only kill bacteria. They do not kill viruses.\n\n14. Thermal scanners can diagnose the coronavirus\n\nThermal scanners can detect whether someone has a fever — which might result from any number of health issues.\n\nSymptoms of COVID-19 can appear 2–14 days after the infection develops. This means that even if a person develops symptoms, they may have a normal temperature for days before a fever begins.\n\n15. Garlic protects against coronaviruses\n\nSome research suggests that garlic may slow the growth of some species of bacteria. COVID-19 results from a virus, not bacteria.\n\n\n\nThere is no evidence that garlic can protect people from COVID-19.\n\n16. Parcels from China can spread the coronavirus\n\nFrom previous research into coronaviruses similar to SARS-CoV-2, including those that cause SARS and MERS, scientists believe that the virus cannot survive on letters or packages for extended periods.\n\nThe CDC explain that “Although the virus can survive for a short period on some surfaces, it is unlikely to be spread from domestic or international mail, products, or packaging.”\n\n17. Home remedies can cure and protect against COVID-19\n\nNo home remedies can protect against COVID-19. This goes for vitamin C, essential oils, silver colloid, sesame oil, garlic, fish tank cleaner, sage, or water, even when a person sips it every 15 minutes.\n\nThe best approach is to wash the hands frequently, for 20 seconds at a time, to use an alcohol-based hand sanitizer, to wear a face covering in public, and to avoid crowded places.\n\n18. You can catch the coronavirus from eating Chinese food in the US\n\nNo, you cannot.\n\n19. You can catch the coronavirus from urine and feces\n\nThis is likely false, but the jury is currently out. According to Prof. John Edmunds, from the London School of Hygiene & Tropical Medicine, in the U.K.:\n\n“It isn’t a very pleasant thought, but every time you swallow, you swallow mucus from your upper respiratory tract. In fact, this is an important defensive mechanism. This sweeps viruses and bacteria down into our gut where they are denatured in the acid conditions of our stomachs.”\n\n“With modern, very highly sensitive detection mechanisms, we can detect these viruses in feces. Usually, the viruses that we can detect in this way are not infectious to others, as they have been destroyed by our guts.”\n\nHowever, it is worth noting that some research suggests that viruses similar to SARS-CoV-2 might persist in feces. A research letter in JAMA also concludes that SARS-CoV-2 is present in feces.\n\n20. The virus will die off when temperatures rise\n\nSome viruses, such as cold and flu viruses, spread more easily in colder months. This does not mean that their transmission stops in warmer weather.\n\nAs it stands, scientists do not know how temperature changes influence the behavior of SARS-CoV-2.\n\n21. The coronavirus is the deadliest virus known to humans\n\nWhile SARS-CoV-2 does appear to be more dangerous than influenza viruses, it is not the deadliest virus that people have faced. Others, such as the Ebola virus, have higher mortality rates.\n\n22. Flu and pneumonia vaccines can protect against COVID-19\n\nBecause SARS-CoV-2 is distinct from other viruses, no existing vaccines can protect against it.\n\n23. The virus originated in a laboratory in China\n\nThere is no evidence to back up this rumor, which has circulated on the internet. As a recent study demonstrates, SARS-CoV-2 is a natural product of evolution.\n\nSome researchers believe that SARS-CoV-2 jumped from pangolins to humans. Others think that it passed to us from bats , like SARS did.\n\n24. The outbreak began because people ate bat soup\n\nWhile scientists are confident that the virus started in animals, there is no evidence that soup was involved.\n\n25. 5G helps SARS-CoV-2 spread\n\nAs the world becomes more connected, some regions are rolling out 5G mobile technology. This has prompted a raft of conspiracy theories.\n\nOne of the most recent to emerge is that 5G is responsible for the swift spread of SARS-CoV-2 across the globe. This is a myth.\n\nSome people believe that 5G helps viruses communicate, often citing a paper from 2011. In this study, the authors conclude that bacteria can communicate via electromagnetic signals.\n\nHowever, experts dispute this theory. In any case, SARS-CoV-2 is a virus, not a bacterium.\n\nWuhan was one of the first cities to trial 5G in China, which helps explain the origin of some of these theories. However, Beijing, Shanghai, and Guangzhou also rolled out 5G at a similar time.\n\nIt is also worth noting that COVID-19 has significantly impacted countries with very little 5G coverage, such as Iran.\n\n26. Drinking alcohol reduces the risk of infection\n\nThe WHO have released a response to the series of myths surrounding alcohol and COVID-19. They explain that while alcohol can disinfect the skin, it does not have this effect inside the body.\n\nThey continue, “Consuming any alcohol poses health risks, but consuming high-strength ethyl alcohol (ethanol), particularly if it has been adulterated with methanol, can result in severe health consequences, including death.”\n\nAlso, in a fact sheet on the subject, they explain that “Alcohol use, especially heavy use, weakens the immune system — and thus reduces the ability to cope with infectious diseases.”\n\nAnd because alcohol is associated with a number of diseases, it may make people more vulnerable to COVID-19.\n\n27. Injecting or consuming bleach or disinfectant kills the virus\n\nConsuming or injecting disinfectant or bleach does not wipe out viruses in the body, and it can be extremely dangerous.\n\nAs Dr. Wayne Carter, an associate professor at the University of Nottingham’s Faculty of Medicine & Health Sciences, in the U.K., explains, “Disinfectants and bleach are strong oxidizing agents, useful to kill bacteria or viruses when they are deposited on surfaces, but these agents should not be ingested or injected. These agents can cause severe tissue burns and blood vessel damage.”\n\nDr. Penny Ward, a visiting professor of pharmaceutical medicine at King’s College London, in the U.K., notes, “Drinking bleach kills. Injecting bleach kills faster.”\n\n28. You can contract the coronavirus in swimming pools\n\nAs the CDC observe, no evidence suggests that SARS-CoV-2 transmits via the water in swimming pools, hot tubs, or water parks. If this water is disinfected with chlorine or bromine, it should inactivate the virus.\n\nStill, the virus can transmit in all the usual ways in these and any other shared spaces. A person contracts a SARS-CoV-2 infection by inhaling respiratory droplets that contain the virus or coming into contact with infected surfaces.\n\nAs in other public places, the CDC recommend staying 6 feet, or 2 meters, away from others at pools and water parks and wearing cloth face coverings when not in the water.\n\nPeople who operate pools should take extra care to clean and disinfect all facilities.\n\nIn a follow-up article, we explore 5 persistent myths about COVID-19 and shed light on the roles of vitamin C, vitamin D, and zinc.\n\n29. If you get a COVID-19 vaccine, you will no longer transmit SARS-CoV-2 to others\n\nThe question as to whether the currently authorized COVID-19 vaccines might stop SARS-CoV-2 transmission altogether has arisen in the context of speculations about “vaccine passports.”\n\nSuch documentation would, in theory, allow people who have had a COVID-19 vaccine to move freely once again.\n\nHowever, at present there is not enough evidence to prove whether or not thecurrently authorized vaccinesstop the spread of the SARS-CoV-2 virus.\n\nThe trial results for the Pfizer-BioNTech , Moderna-NIAID , and Oxford-AstraZeneca vaccines so far suggest they are effective in preventing symptoms of COVID-19.

In [17]:
claim_final.columns

Index(['fact_check_url', 'news_url', 'claim_text', 'label', 'dataset', 'split',
       'label_confidence', 'article_text', 'content_status', 'archive_url',
       'is_archived', 'source_domain', 'is_hydrated', 'fetch_status', 'lang',
       'content_char_len', 'claim_norm_hash', 'ingested_at', 'fetch_attempts',
       'last_fetch_at'],
      dtype='object')

In [18]:
claim_final.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 518 entries, 0 to 517
Data columns (total 20 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   fact_check_url    518 non-null    object
 1   news_url          518 non-null    object
 2   claim_text        518 non-null    object
 3   label             518 non-null    object
 4   dataset           518 non-null    object
 5   split             518 non-null    object
 6   label_confidence  518 non-null    object
 7   article_text      388 non-null    object
 8   content_status    518 non-null    object
 9   archive_url       12 non-null     object
 10  is_archived       518 non-null    bool  
 11  source_domain     518 non-null    object
 12  is_hydrated       518 non-null    bool  
 13  fetch_status      518 non-null    object
 14  lang              388 non-null    object
 15  content_char_len  518 non-null    int64 
 16  claim_norm_hash   518 non-null    object
 17  ingested_at     

In [19]:
claim_final['content_status'].value_counts(normalize=True)

content_status
partial         0.638996
title_only      0.250965
full_article    0.110039
Name: proportion, dtype: float64

In [20]:
claim_final['is_hydrated'].value_counts(normalize=True)

is_hydrated
True     0.749035
False    0.250965
Name: proportion, dtype: float64

In [21]:
claim_final['fetch_status'].value_counts(normalize=True)

fetch_status
success             0.725869
http_404            0.177606
robots_blocked      0.073359
archived_success    0.023166
Name: proportion, dtype: float64

In [22]:
claim_final['is_archived'].value_counts(normalize=True)

is_archived
False    0.976834
True     0.023166
Name: proportion, dtype: float64

In [23]:
claim_final['fetch_attempts'].value_counts(normalize=True)

fetch_attempts
1    0.903475
0    0.073359
2    0.023166
Name: proportion, dtype: float64

### News

In [24]:
news_fake.columns

Index(['Unnamed: 0', 'type', 'fact_check_url', 'archive', 'news_url',
       'news_url2', 'news_url3', 'news_url4', 'news_url5', 'title',
       'newstitle', 'content', 'abstract', 'publish_date', 'meta_keywords'],
      dtype='object')

In [25]:
news_real.columns

Index(['Unnamed: 0', 'type', 'fact_check_url', 'news_url', 'title',
       'newstitle', 'content', 'abstract', 'publish_date', 'meta_keywords'],
      dtype='object')

Fake dataset has extra columns: news_url2-5, archive.

In [26]:
news_real['label'] = 'true'
news_fake['label'] = 'false'

In [27]:
news_fake.head()

,Unnamed: 0,type,fact_check_url,archive,news_url,news_url2,news_url3,news_url4,news_url5,title,newstitle,content,abstract,publish_date,meta_keywords,label
0,0,post,https://factcheck.afp.com/false-advice-refusin...,https://perma.cc/J4N6-39D5,https://www.facebook.com/photo.php?fbid=551960...,NaN,NaN,NaN,NaN,Facebook posts shared in at least three countr...,NaN,NaN,NaN,NaN,NaN,false
1,1,article,https://www.politifact.com/factchecks/2020/apr...,NaN,http://legis.wisconsin.gov/assembly/republican...,NaN,NaN,NaN,NaN,Wisconsin is Òclearly seeing a decline in COVI...,"""Wisconsin Legislature Takes Gov. Evers to Court""",speaker robin vos r rochester and senate major...,NaN,NaN,"""""",false
2,2,post,https://factcheck.afp.com/posts-claim-children...,https://perma.cc/V4HX-M2XJ,https://www.facebook.com/iAmJessenia/photos/a....,NaN,NaN,NaN,NaN,Facebook posts claim a child who is infected w...,NaN,NaN,NaN,NaN,NaN,false
3,3,post,https://checkyourfact.com/2020/04/20/fact-chec...,NaN,https://www.facebook.com/kokernagnews/photos/a...,NaN,NaN,NaN,NaN,IndiaÕs Ministry of Home Affairs banning citiz...,NaN,NaN,NaN,NaN,NaN,false
4,5,post,https://checkyourfact.com/2020/04/20/fact-chec...,NaN,https://www.facebook.com/photo.php?fbid=102193...,NaN,NaN,NaN,NaN,"42 Democratic senators, plus two Independents,...",NaN,NaN,NaN,NaN,NaN,false


In [28]:
news_fake.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 925 entries, 0 to 924
Data columns (total 16 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   Unnamed: 0      925 non-null    int64 
 1   type            885 non-null    object
 2   fact_check_url  925 non-null    object
 3   archive         377 non-null    object
 4   news_url        885 non-null    object
 5   news_url2       93 non-null     object
 6   news_url3       60 non-null     object
 7   news_url4       34 non-null     object
 8   news_url5       15 non-null     object
 9   title           925 non-null    object
 10  newstitle       467 non-null    object
 11  content         416 non-null    object
 12  abstract        269 non-null    object
 13  publish_date    145 non-null    object
 14  meta_keywords   467 non-null    object
 15  label           925 non-null    object
dtypes: int64(1), object(15)
memory usage: 115.8+ KB


Remove urls2-5 as mostly empty
check if newstitle is null when title isnt and vice versa
check content, abstract, publish_date, meta_keywords.

In [29]:
news_fake.drop(columns=['news_url2','news_url3','news_url4','news_url5'],inplace=True,axis=1)

In [30]:
news_fake

,Unnamed: 0,type,fact_check_url,archive,news_url,title,newstitle,content,abstract,publish_date,meta_keywords,label
0,0,post,https://factcheck.afp.com/false-advice-refusin...,https://perma.cc/J4N6-39D5,https://www.facebook.com/photo.php?fbid=551960...,Facebook posts shared in at least three countr...,NaN,NaN,NaN,NaN,NaN,false
1,1,article,https://www.politifact.com/factchecks/2020/apr...,NaN,http://legis.wisconsin.gov/assembly/republican...,Wisconsin is Òclearly seeing a decline in COVI...,"""Wisconsin Legislature Takes Gov. Evers to Court""",speaker robin vos r rochester and senate major...,NaN,NaN,"""""",false
2,2,post,https://factcheck.afp.com/posts-claim-children...,https://perma.cc/V4HX-M2XJ,https://www.facebook.com/iAmJessenia/photos/a....,Facebook posts claim a child who is infected w...,NaN,NaN,NaN,NaN,NaN,false
3,3,post,https://checkyourfact.com/2020/04/20/fact-chec...,NaN,https://www.facebook.com/kokernagnews/photos/a...,IndiaÕs Ministry of Home Affairs banning citiz...,NaN,NaN,NaN,NaN,NaN,false
4,5,post,https://checkyourfact.com/2020/04/20/fact-chec...,NaN,https://www.facebook.com/photo.php?fbid=102193...,"42 Democratic senators, plus two Independents,...",NaN,NaN,NaN,NaN,NaN,false
...,...,...,...,...,...,...,...,...,...,...,...,...
920,971,post,https://healthfeedback.org/claimreview/the-vir...,https://archive.is/8cPJC,https://www.youtube.com/watch?v=qFlqXPl_hZQ&fe...,Genetic evidence within the Spike gene of the ...,"""Coronavirus whistleblower speaks out about po...",NaN,dr. li meng yan joins tucker carlson with insi...,9/15/20,"""coronavirus whistleblower, fox news whistlebl...",false
921,972,post,https://healthfeedback.org/claimreview/cloth-m...,https://archive.is/Qca7p,https://www.facebook.com/108082977404530/posts...,Cloth masks cannot block smoke particles which...,"""Log In or Sign Up to View""",do you want to join facebook ?.,NaN,NaN,"""""",false
922,973,post,https://www.factcheck.org/2020/09/viral-post-f...,NaN,https://www.facebook.com/fred.childs.5/posts/1...,PGA golfer Bubba Wallace wrote Facebook post t...,"""Fred Childs""",see more of fred childs on facebook.,hooray for bubba watson. he put into words wha...,NaN,"""""",false
923,974,post,https://healthfeedback.org/claimreview/covid-1...,https://archive.vn/VAdYH,https://www.facebook.com/BenSwannRealityCheck/...,The World Integrated Trade Solutions (WITS) we...,"""Ben Swann""",part of what infuriates trump supporters is th...,this is incredibly strange. world bank website...,NaN,"""""",false


In [31]:
news_fake.columns

Index(['Unnamed: 0', 'type', 'fact_check_url', 'archive', 'news_url', 'title',
       'newstitle', 'content', 'abstract', 'publish_date', 'meta_keywords',
       'label'],
      dtype='object')

In [32]:
news_real.columns

Index(['Unnamed: 0', 'type', 'fact_check_url', 'news_url', 'title',
       'newstitle', 'content', 'abstract', 'publish_date', 'meta_keywords',
       'label'],
      dtype='object')

In [33]:
news_fake.drop(columns=['archive'], inplace=True)

In [34]:
news = pd.concat([news_fake,news_real],axis=0,ignore_index=True)
news.head()

,Unnamed: 0,type,fact_check_url,news_url,title,newstitle,content,abstract,publish_date,meta_keywords,label
0,0,post,https://factcheck.afp.com/false-advice-refusin...,https://www.facebook.com/photo.php?fbid=551960...,Facebook posts shared in at least three countr...,NaN,NaN,NaN,NaN,NaN,false
1,1,article,https://www.politifact.com/factchecks/2020/apr...,http://legis.wisconsin.gov/assembly/republican...,Wisconsin is Òclearly seeing a decline in COVI...,"""Wisconsin Legislature Takes Gov. Evers to Court""",speaker robin vos r rochester and senate major...,NaN,NaN,"""""",false
2,2,post,https://factcheck.afp.com/posts-claim-children...,https://www.facebook.com/iAmJessenia/photos/a....,Facebook posts claim a child who is infected w...,NaN,NaN,NaN,NaN,NaN,false
3,3,post,https://checkyourfact.com/2020/04/20/fact-chec...,https://www.facebook.com/kokernagnews/photos/a...,IndiaÕs Ministry of Home Affairs banning citiz...,NaN,NaN,NaN,NaN,NaN,false
4,5,post,https://checkyourfact.com/2020/04/20/fact-chec...,https://www.facebook.com/photo.php?fbid=102193...,"42 Democratic senators, plus two Independents,...",NaN,NaN,NaN,NaN,NaN,false


In [36]:
def news_schema(
    df: pd.DataFrame,
    split: str = "train",
    dataset: str = "coaid",
    title_col: str = "title",
    url_col: str = "news_url",
    max_workers: int = 12,
    batch_size = None,
    save_batches_dir = None,
    throttle_seconds: float = 1.0,
    show_progress: bool = True
) -> pd.DataFrame:
    """
    Prepare the news-style DataFrame for the unified schema:
      - preserve all original columns,
      - drop a few legacy columns (safely),
      - add only the URL/article-related columns produced by threaded_hydrate,
      - avoid overwriting any existing original columns.

    Assumes threaded_hydrate (and hydrate_claim) are importable from article_scraping.
    """
    temp = df.copy()

    # metadata
    temp["dataset"] = dataset
    temp["split"] = split

    # drop legacy columns (safe: ignore errors if cols don't exist)
    temp.drop(columns=["Unnamed: 0", "publish_date", "abstract", "meta_keywords"], inplace=True, errors="ignore")

    # set label confidence
    temp["label_confidence"] = "gold"

    # Run the threaded hydrator to get url/article metadata (res_df indexed by original index)
    res_df = threaded_hydrate(
        temp,
        title_col=title_col,
        url_col=url_col,
        hydrate_fn=None,
        dataset=dataset,
        max_workers=max_workers,
        batch_size=batch_size,
        save_batches_dir=save_batches_dir,
        throttle_seconds=throttle_seconds,
        show_progress=show_progress,
    )

    # expected keys produced by hydrate_claim (defensive)
    expected_cols = [
        "article_text",
        "content_status",
        "news_url",
        "archive_url",
        "is_archived",
        "source_domain",
        "is_hydrated",
        "fetch_status",
        "lang",
        "content_char_len",
        "claim_norm_hash",
        "ingested_at",
        "fetch_attempts",
        "last_fetch_at",
    ]

    # select only expected & available columns
    available_cols = [c for c in expected_cols if c in res_df.columns]

    # IMPORTANT: do not overwrite any original columns in temp.
    cols_to_add = [c for c in available_cols if c not in temp.columns]

    # align and join only the new columns
    res_df = res_df.reindex(temp.index)
    out = temp.join(res_df[cols_to_add], how="left")

    # Fill sensible defaults for the newly added columns (if present)
    if "content_status" in out.columns:
        out["content_status"] = out["content_status"].fillna("title_only")
    if "fetch_status" in out.columns:
        out["fetch_status"] = out["fetch_status"].fillna("not_fetched")
    if "content_char_len" in out.columns:
        out["content_char_len"] = out["content_char_len"].fillna(0).astype(int)
    if "is_hydrated" in out.columns:
        out["is_hydrated"] = out["is_hydrated"].fillna(False).astype(bool)
    if "is_archived" in out.columns:
        out["is_archived"] = out["is_archived"].fillna(False).astype(bool)
    if "source_domain" in out.columns:
        out["source_domain"] = out["source_domain"].fillna("")

    out.rename(columns={"title": "claim_text"}, inplace=True)
    return out

In [37]:
news.tail()

,Unnamed: 0,type,fact_check_url,news_url,title,newstitle,content,abstract,publish_date,meta_keywords,label
5452,10078,article,https://www.politifact.com/factchecks/2020/oct...,https://www.themonitor.com/2020/10/05/kamala-h...,Says the “The Rio Grande Valley is 4.7% of the...,"""403 Forbidden""",NaN,NaN,NaN,"""""",true
5453,10079,article,https://www.politifact.com/factchecks/2020/oct...,https://www.wral.com/cooper-forest-engage-in-l...,"Georgia has “almost 100,000 more (COVID-19) ca...","""Cooper, Forest engage in lone gubernatorial d...",governor dan forest. live debate at u. n c. t ...,democratic gov. roy cooper and republican lt. ...,2020-10-14T18:47:00-04:00,"""Roy Cooper,Dan Forest,2020 governors race,deb...",true
5454,10080,post,https://www.politifact.com/factchecks/2020/oct...,https://www.facebook.com/photo.php?fbid=402154...,“Wisconsin Republicans have not passed a singl...,"""Facebook""",NaN,NaN,NaN,"""""",true
5455,10081,post,https://www.politifact.com/factchecks/2020/sep...,https://twitter.com/SenChrisLarson/status/1308...,“(Republicans) have the power to overturn the ...,"""""",this browser is no longer supported. please sw...,NaN,NaN,"""""",true
5456,10082,article,https://www.politifact.com/factchecks/2020/sep...,https://medium.com/@calebrowden/caleb-rowden-c...,“Over three months after receiving CARES Act r...,"""Caleb Rowden Calls on Boone County to Expedit...",coronavirus pandemic is exacting heavy toll on...,coronavirus pandemic is exacting heavy toll on...,2020-09-04T21:12:03.130Z,"""""",true


In [ ]:
news_final = news_schema(news,'train','coaid')

In [41]:
news_final.head()

,fact_check_url,news_url,claim_text,label,dataset,split,label_confidence,article_text,content_status,archive_url,is_archived,source_domain,is_hydrated,fetch_status,lang,content_char_len,claim_norm_hash,ingested_at,fetch_attempts,last_fetch_at
0,medicalnewstoday.com,https://www.medicalnewstoday.com/articles/coro...,"""Spraying chlorine or alcohol on the skin kill...",false,coaid,train,gold,Share on Pinterest Design by Andrew Nguyen\n\n...,partial,None,False,www.medicalnewstoday.com,True,success,en,14363,3c009e65f139be1e301f16445d3f4b629c318438,2025-11-10T20:33:40.838323+00:00,1,2025-11-10T20:33:42.138654+00:00
1,medicalnewstoday.com,https://www.medicalnewstoday.com/articles/coro...,"""Only older adults and young people are at risk""",false,coaid,train,gold,Share on Pinterest Design by Andrew Nguyen\n\n...,partial,None,False,www.medicalnewstoday.com,True,success,en,14363,cc07271af27f0283b641237511bdebd97233a76f,2025-11-10T20:33:41.842831+00:00,1,2025-11-10T20:33:43.850471+00:00
2,medicalnewstoday.com,https://www.medicalnewstoday.com/articles/coro...,"""Children cannot get COVID-19""",false,coaid,train,gold,Share on Pinterest Design by Andrew Nguyen\n\n...,partial,None,False,www.medicalnewstoday.com,True,success,en,14363,b68387a76bf777c9b398e22fdc437cf097764b9e,2025-11-10T20:33:41.841018+00:00,1,2025-11-10T20:33:43.913203+00:00
3,medicalnewstoday.com,https://www.medicalnewstoday.com/articles/coro...,"""COVID-19 is just like the flu""",false,coaid,train,gold,Share on Pinterest Design by Andrew Nguyen\n\n...,partial,None,False,www.medicalnewstoday.com,True,success,en,14363,83c391e0dbefe0b59203d12b038ad48ec2c7e772,2025-11-10T20:33:41.847272+00:00,1,2025-11-10T20:33:43.851939+00:00
4,medicalnewstoday.com,https://www.medicalnewstoday.com/articles/coro...,"""Everyone with COVID-19 dies""",false,coaid,train,gold,Share on Pinterest Design by Andrew Nguyen\n\n...,partial,None,False,www.medicalnewstoday.com,True,success,en,14363,dcb79cec8ac3c34201f1c4e7d892606586eff268,2025-11-10T20:33:41.841580+00:00,1,2025-11-10T20:33:43.691045+00:00


In [ ]:
claim_final.to_parquet("unified_coaid_claims.parquet")
news_final.to_parquet("unified_coaid_news.parquet")